# DSAR × Lakeflow Declarative Pipelines · 01b · CDC variant (SCD1 silver + Auto Loader)

Same as `01_sdp_pipeline`, **except silver is an SCD type-1 dimension** maintained
by **AUTO CDC** (`dp.create_auto_cdc_flow`) keyed on `user_id`. This faithfully
matches Allegiant's real Merlot pipeline (`dbo_user_silver_cdc`) — the exact flow
whose fatal `append-only source` failure started this whole effort.

**Attach as a *separate* Lakeflow pipeline** with its **own target schema** (config
`dsar.schema` = e.g. `allegiant_air_sdp_dsar_cdc`) so it doesn't collide with the
append-silver variant. Modern API throughout.

```
/Volumes/.../raw_user/landing/*.json
        │  Auto Loader (cloudFiles)
        ▼
raw_user ─stream─▶ bronze_user ─stream─▶ (bronze_user_cdc_feed) ═AUTO CDC═▶ silver_user ─batch MV─▶ gold_user
 (ingest)          mask PII              change feed for CDC        SCD type 1        aggregate
```

> Run `00` against **this variant's schema** first (so the volume + landing files
> exist there). The `bronze_user_cdc_feed` view is **required** by AUTO CDC (it needs
> a streaming change source). `skipChangeCommits` is set on every streaming read so
> an erasure never breaks the pipeline.


## 0. Config


In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

def cfg(key, default):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

CATALOG = cfg("dsar.catalog", "dkushari_uc")
SCHEMA  = cfg("dsar.schema",  "allegiant_air_sdp_dsar_cdc")
VOLUME  = cfg("dsar.volume",  "raw_user")
FQ      = f"{CATALOG}.{SCHEMA}"
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/landing"   # Auto Loader source (initial/ + incremental/)
print("Auto Loader landing:", LANDING, "| target schema:", FQ)


## 1. Raw — Auto Loader ingest from the volume (streaming)\n\nIdentical to the primary variant — ingests landing JSON into `raw_user`.


In [ ]:
# Explicit schema => deterministic, no _rescued_data drift, no schemaLocation needed.
RAW_SCHEMA = ("event_id string, user_id string, email string, full_name string, "
              "profile_json string, revenue double, event_ts string, _ingest_ts string")

@dp.table(
    name="raw_user",
    comment="Auto Loader ingest of landing JSON files (cleartext PII). Recurses landing/ so both initial/ and incremental/ files are picked up.",
    table_properties={"quality": "raw"},
)
def raw_user():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .schema(RAW_SCHEMA)
            .load(LANDING))


## 2. Bronze — mask PII inline (streaming)\n\nIdentical to the primary variant.


In [ ]:
REDACT = "***REDACTED***"

def _mask_json(col):
    # in-JSON masking, native SQL — quote-anchored keys so "name" != "appName"
    e = f"regexp_replace({col}, '(\"email\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    e = f"regexp_replace({e}, '(\"name\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    return e

@dp.table(
    name="bronze_user",
    comment="Streaming bronze: PII masked inline; user_id/revenue preserved.",
    table_properties={"quality": "bronze"},
)
def bronze_user():
    src = (spark.readStream
           .option("skipChangeCommits", "true")   # survive erasure on raw_user
           .table(f"{FQ}.raw_user"))
    return src.select(
        "event_id",
        "user_id",                                    # stable key — NOT masked
        F.lit(REDACT).alias("email"),                 # PII -> redact
        F.lit(REDACT).alias("full_name"),             # PII -> redact
        F.expr(_mask_json("profile_json")).alias("profile_json"),  # in-JSON PII -> redact
        "revenue", "event_ts", "_ingest_ts",          # non-PII -> preserve
    )


## 3. Silver — SCD type 1 via AUTO CDC

`dp.create_streaming_table` declares the target; a streaming **change feed** view
over bronze provides the CDC source; `dp.create_auto_cdc_flow` applies it as SCD
type 1 (one row per `user_id`, latest wins by `_ingest_ts`).

`skipChangeCommits` on the change-feed read of bronze is what keeps this flow from
failing when an erasure deletes from bronze — the fix for the original incident.


In [ ]:
dp.create_streaming_table(
    name="silver_user",
    comment="SCD type-1 customer dimension via AUTO CDC, keyed on user_id.",
    table_properties={"quality": "silver"},
)

@dp.temporary_view(name="bronze_user_cdc_feed")
def bronze_user_cdc_feed():
    # streaming change source for AUTO CDC; skipChangeCommits => an erasure DELETE
    # on bronze is skipped here (02 deletes silver explicitly), keeping the flow alive.
    return (spark.readStream
            .option("skipChangeCommits", "true")
            .table(f"{FQ}.bronze_user"))

dp.create_auto_cdc_flow(
    target="silver_user",
    source="bronze_user_cdc_feed",
    keys=["user_id"],
    sequence_by=F.col("_ingest_ts"),
    stored_as_scd_type=1,
)


## 4. Gold — per-customer aggregate (materialized view)

Reads the SCD1 **silver** dimension. Batch recompute, so erasures propagate on
refresh; you cannot DELETE from it (see `02`).


In [ ]:
@dp.materialized_view(
    name="gold_user",
    comment="Per-customer rollup over the SCD1 silver dimension, recomputed each refresh.",
    table_properties={"quality": "gold"},
)
def gold_user():
    return (spark.read.table(f"{FQ}.silver_user")
            .groupBy("user_id")
            .agg(F.sum("revenue").alias("lifetime_revenue"),
                 F.count("*").alias("event_count"),
                 F.max("event_ts").alias("last_event_ts")))
